# 01 — Revisión de datos

Objetivo: determinar qué información de Volve está disponible y qué condiciones requieren revisión antes de explorar o modelar. No se modifican originales ni se aplica limpieza automática.

**Recorrido:** carga y contexto → panorama con YData Profiling → validaciones por pozo y flujo → síntesis de hallazgos → próximos pasos en el notebook 02.

El profiling orienta la revisión. Los controles específicos permiten interpretar sus señales en el contexto operacional; las decisiones de limpieza se documentan al final.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data" / "bronze").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "bronze").is_dir(), "Abrir desde la raíz del proyecto o notebooks/."
ARCHIVO = ROOT / "data" / "bronze" / "Volve production data.xlsx"

# Los identificadores se conservan como texto.
diario = pd.read_excel(ARCHIVO, sheet_name="Daily Production Data", dtype={
    "NPD_WELL_BORE_CODE": "string", "NPD_FIELD_CODE": "string", "NPD_FACILITY_CODE": "string"
})
diario["DATEPRD"] = pd.to_datetime(diario["DATEPRD"], errors="raise")
mensual = pd.read_excel(ARCHIVO, sheet_name="Monthly Production Data", skiprows=[1],
                        na_values=["NULL"], dtype={"NPDCode": "string"})


## Contenido y estructura

In [2]:
print(f"Diario: {diario.shape[0]:,} registros, {diario.shape[1]} columnas")
print(f"Mensual: {mensual.shape[0]:,} registros, {mensual.shape[1]} columnas")
print("Período diario:", diario["DATEPRD"].min(), "a", diario["DATEPRD"].max())
display(diario.head())
display(mensual.head())
display(diario.dtypes.rename("tipo").to_frame())

Diario: 15,634 registros, 24 columnas
Mensual: 526 registros, 10 columnas
Período diario: 2007-09-01 00:00:00 a 2016-12-01 00:00:00


,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


,Wellbore name,NPDCode,Year,Month,On Stream,Oil,Gas,Water,GI,WI
0,15/9-F-1 C,7405,2014,4,227.50000,11142.47,1597936.65,0.00,NaN,NaN
1,15/9-F-1 C,7405,2014,5,733.83334,24901.95,3496229.65,783.48,NaN,NaN
2,15/9-F-1 C,7405,2014,6,705.91666,19617.76,2886661.69,2068.48,NaN,NaN
3,15/9-F-1 C,7405,2014,7,742.41666,15085.68,2249365.75,6243.98,NaN,NaN
4,15/9-F-1 C,7405,2014,8,432.99166,6970.43,1048190.80,4529.75,NaN,NaN


,tipo
DATEPRD,datetime64[ns]
WELL_BORE_CODE,object
NPD_WELL_BORE_CODE,string[python]
NPD_WELL_BORE_NAME,object
NPD_FIELD_CODE,string[python]
NPD_FIELD_NAME,object
NPD_FACILITY_CODE,string[python]
NPD_FACILITY_NAME,object
ON_STREAM_HRS,float64
AVG_DOWNHOLE_PRESSURE,float64


## Diccionario original

Consultar el estado de verificación y las observaciones antes de asumir unidades.

In [3]:
diccionario = pd.read_excel(ROOT / "data" / "bronze" / "Diccionario_Volve.xlsx", sheet_name=None, header=3)
for hoja, tabla in diccionario.items():
    print(hoja)
    display(tabla.dropna(how="all"))

Diario


,Variable original,Descripción en español,Tipo de dato,Unidad,Fuente,Estado de verificación,Observaciones
0,DATEPRD,Fecha del registro diario,Fecha,Día,"Excel, Daily Production Data",Observado en archivo,No se verificó horario de corte ni zona horaria.
1,WELL_BORE_CODE,Identificador textual del pozo o sondeo,Texto,No aplica,"Excel, Daily Production Data",Interpretación por verificar,Ejemplo: NO 15/9-F-1 C. Preservar como identif...
2,NPD_WELL_BORE_CODE,Código del sondeo asociado a NPD,Texto (código),No aplica,"Excel, Daily Production Data",Interpretación por verificar,Códigos numéricos almacenados como texto; veri...
3,NPD_WELL_BORE_NAME,Nombre del sondeo asociado a NPD,Texto,No aplica,"Excel, Daily Production Data",Observado en archivo,Ejemplo: 15/9-F-1 C.
4,NPD_FIELD_CODE,Código del campo asociado a NPD,Texto (código),No aplica,"Excel, Daily Production Data",Interpretación por verificar,Valor observado: 3420717. No tratar como magni...
5,NPD_FIELD_NAME,Nombre del campo,Texto,No aplica,"Excel, Daily Production Data",Observado en archivo,Valor observado: VOLVE.
6,NPD_FACILITY_CODE,Código de instalación asociado a NPD,Texto (código),No aplica,"Excel, Daily Production Data",Interpretación por verificar,Valor observado: 369304. Verificar catálogo of...
7,NPD_FACILITY_NAME,Nombre de la instalación,Texto,No aplica,"Excel, Daily Production Data",Observado en archivo,Valor observado: MÆRSK INSPIRER.
8,ON_STREAM_HRS,Horas en operación durante el registro,Número decimal,Horas,"Artículo, tabla 1 (S2); Excel mensual",Definición respaldada; criterio pendiente,Aplica a operación de producción/inyección. Ha...
9,AVG_DOWNHOLE_PRESSURE,Presión promedio de fondo,Número decimal,Pendiente de confirmar,"Artículo, tabla 1 (S2)",Definición respaldada; unidad pendiente,Verificar unidad y período de promediado. En p...


Mensual


,Variable original,Descripción en español,Tipo de dato,Unidad,Fuente,Estado de verificación,Observaciones
0,Wellbore name,Nombre del sondeo,Texto,No aplica,"Excel, Monthly Production Data, filas 1–2",Observado en archivo,Relacionar con NPD_WELL_BORE_NAME de la hoja d...
1,NPDCode,Código NPD del sondeo (interpretación),Entero usado como código,No aplica,"Excel, Monthly Production Data, filas 1–2",Interpretación por verificar,"Para cruces, normalizar el tipo con NPD_WELL_B..."
2,Year,Año del registro mensual,Entero,Año,"Excel, Monthly Production Data, filas 1–2",Observado en archivo,Combinar con Month para construir el período.
3,Month,Mes del registro mensual,Entero,Mes,"Excel, Monthly Production Data, filas 1–2",Observado en archivo,Validar rango 1 a 12.
4,On Stream,Tiempo de operación mensual,Número decimal,hrs,"Excel, Monthly Production Data, filas 1–2",Unidad explícita; criterio pendiente,No está expresado en días. Verificar criterio ...
5,Oil,Volumen mensual de petróleo,Número decimal,Sm3,"Excel, Monthly Production Data, filas 1–2",Unidad explícita en archivo,Metros cúbicos estándar; temperatura y presión...
6,Gas,Volumen mensual de gas,Número decimal,Sm3,"Excel, Monthly Production Data, filas 1–2",Unidad explícita en archivo,Metros cúbicos estándar; temperatura y presión...
7,Water,Volumen mensual de agua,Número decimal,Sm3,"Excel, Monthly Production Data, filas 1–2",Unidad explícita en archivo,Metros cúbicos estándar; temperatura y presión...
8,GI,Volumen de gas inyectado (interpretación),Número o texto NULL,Sm3,"Excel, Monthly Production Data, filas 1–2",Unidad explícita; significado pendiente,Confirmar expansión de GI. NULL es texto y deb...
9,WI,Volumen de agua inyectada (interpretación),Número o texto NULL,Sm3,"Excel, Monthly Production Data, filas 1–2",Unidad explícita; significado pendiente,Confirmar expansión de WI. NULL es texto y deb...


## Reportes automáticos con YData Profiling

Ejecutar después de cargar los datos. Se genera un HTML por flujo, con todos sus registros originales, sin imputar ni eliminar filas. Producción e inyección se perfilan separadamente porque sus variables disponibles difieren.

Los reportes incluyen distribuciones, faltantes, correlaciones y alertas automáticas. Una alerta no demuestra un error ni justifica eliminar una variable. Las correlaciones agrupan pozos y períodos: no representan efectos causales ni sustituyen el análisis temporal por pozo del notebook 02. Las unidades pendientes siguen pendientes.

Los HTML se guardan en `outputs/` y se abren en el navegador, sin incrustarlos en el notebook. Se regeneran al ejecutar esta sección; si cambian los datos, volver a ejecutarla. Instalar antes `requirements.txt` en el entorno del kernel.


In [ ]:
from ydata_profiling import ProfileReport

CARPETA_REPORTES = ROOT / "outputs"
CARPETA_REPORTES.mkdir(exist_ok=True)

for flujo in ("production", "injection"):
    datos_flujo = diario.loc[diario["FLOW_KIND"].eq(flujo)].copy()
    if datos_flujo.empty:
        print(f"Sin registros para {flujo}; no se genera reporte.")
        continue
    # Los códigos identifican entidades: no deben inferirse como magnitudes.
    categoricas = datos_flujo.select_dtypes(include=["object", "string"]).columns
    reporte = ProfileReport(
        datos_flujo,
        title=f"WELLWISE-X — Volve — {flujo}",
        type_schema={columna: "categorical" for columna in categoricas},
        pool_size=1,
        progress_bar=False,
        correlations={"auto": {"calculate": False}, "pearson": {"calculate": True}},
        interactions={"continuous": False},
        missing_diagrams={"heatmap": False},
        dataset={"description": (
            f"Registros diarios de flujo {flujo}, todos los pozos y períodos. "
            "Sin limpieza ni imputación. Unidades diarias pendientes de verificación. "
            "Correlaciones de Pearson descriptivas; no causales."
        )},
    )
    destino = CARPETA_REPORTES / f"volve_{flujo}_profiling.html"
    reporte.to_file(destino)
    print(f"{flujo}: {len(datos_flujo):,} registros — {destino}")


Abrir los archivos desde el explorador: `outputs/volve_production_profiling.html` y `outputs/volve_injection_profiling.html`.

Para revisar primero: alertas, variables con faltantes, ceros y distribuciones. Luego contrastar con los controles por pozo de este notebook. Se desactivaron las interacciones gráficas masivas y el mapa de calor de faltantes para acotar el tiempo de generación; se conservan las correlaciones de Pearson y los resúmenes de faltantes.


## Del profiling a las validaciones

Abrir los HTML anteriores y usar esta guía para interpretar sus señales. La tabla es una guía de lectura, no una extracción automática de las alertas emitidas por YData Profiling.

| Señal a revisar en el reporte | Verificación en este notebook | Criterio de interpretación |
|---|---|---|
| Faltantes | Comparar total, producción y disponibilidad por pozo | La ausencia puede depender del flujo o del sensor; no imputar en bloque. |
| Ceros y extremos en horas o volúmenes | Revisar horas fuera de rango y petróleo positivo con horas cero | Distinguir posibles paradas de inconsistencias; no eliminar automáticamente. |
| Fechas y duplicados | Validar clave pozo-fecha y días ausentes | Los duplicados de filas del profiler no equivalen a duplicados de la clave; los huecos requieren un control temporal. |
| Categorías de flujo y tipo de pozo | Cruzar `FLOW_KIND` y `WELL_TYPE` | Revisar combinaciones que requieren contexto operacional. |
| Correlaciones | Contrastar por pozo y período en el notebook 02 | Quedan pendientes de interpretación; una correlación global no implica causalidad. |

Los controles siguientes se ejecutan aunque el profiler no emita una alerta: responden a requisitos concretos de estos datos.


## Cobertura por pozo y tipo de flujo

In [4]:
display(diario.groupby("NPD_WELL_BORE_NAME").agg(
    registros=("DATEPRD", "size"), inicio=("DATEPRD", "min"), fin=("DATEPRD", "max")
))
display(pd.crosstab(diario["FLOW_KIND"], diario["WELL_TYPE"]))
print("Duplicados por pozo y fecha:", diario.duplicated(["NPD_WELL_BORE_CODE", "DATEPRD"]).sum())
huecos = diario.groupby("NPD_WELL_BORE_NAME")["DATEPRD"].agg(
    lambda fechas: len(pd.date_range(fechas.min(), fechas.max(), freq="D")) - fechas.nunique()
)
display(huecos.rename("dias_ausentes_entre_inicio_y_fin").to_frame())

,registros,inicio,fin
NPD_WELL_BORE_NAME,,,
15/9-F-1 C,746,2014-04-07,2016-04-21
15/9-F-11,1165,2013-07-08,2016-09-17
15/9-F-12,3056,2008-02-12,2016-09-17
15/9-F-14,3056,2008-02-12,2016-09-17
15/9-F-15 D,978,2014-01-12,2016-09-17
15/9-F-4,3327,2007-09-01,2016-12-01
15/9-F-5,3306,2007-09-01,2016-09-18


WELL_TYPE,OP,WI
FLOW_KIND,,
injection,0,6473
production,9143,18


Duplicados por pozo y fecha: 0


,dias_ausentes_entre_inicio_y_fin
NPD_WELL_BORE_NAME,
15/9-F-1 C,0
15/9-F-11,3
15/9-F-12,85
15/9-F-14,85
15/9-F-15 D,2
15/9-F-4,53
15/9-F-5,0


## Faltantes: interpretar las señales del profiling

Los reportes separan producción e inyección. Aquí profundizamos en la disponibilidad por pozo para distinguir faltantes asociados al flujo de sensores con poca cobertura. Los porcentajes globales no bastan para decidir una imputación.


In [ ]:
produccion = diario.loc[diario["FLOW_KIND"].eq("production")].copy()
display(pd.DataFrame({
    "faltantes_total_pct": diario.isna().mean() * 100,
    "faltantes_produccion_pct": produccion.isna().mean() * 100
}).round(2))
variables = ["AVG_DOWNHOLE_PRESSURE", "AVG_ANNULUS_PRESS", "AVG_CHOKE_SIZE_P", "BORE_OIL_VOL"]
display(produccion.groupby("NPD_WELL_BORE_NAME")[variables].agg(lambda s: s.isna().mean() * 100).round(2))

## Registros para revisar

Estas señales no prueban errores. Las horas cero pueden corresponder a paradas; los valores mayores a 24 requieren verificar el criterio de registro.

In [5]:
controles = pd.DataFrame({
    "horas_cero": diario["ON_STREAM_HRS"].eq(0),
    "horas_fuera_0_24": diario["ON_STREAM_HRS"].lt(0) | diario["ON_STREAM_HRS"].gt(24),
    "petroleo_con_horas_cero": diario["BORE_OIL_VOL"].gt(0) & diario["ON_STREAM_HRS"].eq(0),
    "production_con_tipo_WI": diario["FLOW_KIND"].eq("production") & diario["WELL_TYPE"].eq("WI")
})
display(controles.sum().rename("registros").to_frame())
display(diario.loc[controles.drop(columns="horas_cero").any(axis=1)])

,registros
horas_cero,1952
horas_fuera_0_24,20
petroleo_con_horas_cero,1
production_con_tipo_WI,18


,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.00000,0.000000,...,%,0.000000,0.000000,0.000000,0.00,0.00,0.00,NaN,production,WI
91,2014-07-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,215.070057,...,%,52.663308,54.966429,24.233500,522.00,77846.76,154.47,NaN,production,WI
202,2014-10-26,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,220.957970,...,%,36.554275,47.287813,7.917633,333.14,51384.55,336.05,NaN,production,OP
856,2013-10-27,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,24.30839,256.047878,...,%,82.213141,60.017753,53.861978,1219.94,188361.38,69.99,NaN,production,OP
1218,2014-10-26,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,220.578730,...,%,52.942725,63.461084,24.513722,1110.58,171297.30,339.89,NaN,production,OP
1301,2015-01-17,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,0.00000,219.800000,...,%,50.400000,64.800000,21.000000,1026.57,150773.50,461.02,NaN,production,OP
2166,2008-10-26,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,233.664638,...,%,70.283581,75.273871,37.889267,3836.99,547535.29,12.84,NaN,production,OP
2527,2009-10-25,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,24.83333,261.865161,...,%,77.557616,75.845229,45.036169,4190.37,585516.23,279.62,NaN,production,OP
2891,2010-10-31,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,0.000000,...,%,52.641503,88.527274,20.099256,1677.68,248776.29,3676.04,NaN,production,OP
3912,2013-10-27,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,24.14167,0.000000,...,%,33.458990,89.321837,4.623115,280.98,43383.87,4798.50,NaN,production,OP


## Síntesis de la revisión

Esta tabla conecta las señales revisadas con evidencia calculada y el paso siguiente. Se actualiza al ejecutar el notebook; no es una transcripción de las alertas del profiler ni una autorización para limpiar datos. Las unidades y correlaciones siguen pendientes de interpretación.


In [ ]:
faltantes_anular = produccion.groupby("NPD_WELL_BORE_NAME")["AVG_ANNULUS_PRESS"].apply(
    lambda s: s.isna().mean() * 100
)
pozo_mayor_faltante = faltantes_anular.idxmax()
horas_produccion = produccion["ON_STREAM_HRS"]

sintesis = pd.DataFrame([
    {
        "Señal": "Faltantes de petróleo",
        "Evidencia": (
            f"Total: {diario['BORE_OIL_VOL'].isna().mean():.1%}; "
            f"producción: {produccion['BORE_OIL_VOL'].isna().mean():.1%}."
        ),
        "Interpretación / siguiente paso": "Interpretar por flujo; no completar con cero campos que pueden no aplicar."
    },
    {
        "Señal": "Disponibilidad de presión anular",
        "Evidencia": f"Mayor faltante en producción: {pozo_mayor_faltante}, {faltantes_anular.max():.1f}%.",
        "Interpretación / siguiente paso": "Revisar cobertura del sensor por período antes de seleccionar variables."
    },
    {
        "Señal": "Horas operativas",
        "Evidencia": (
            f"En producción: {horas_produccion.eq(0).sum()} registros con cero horas; "
            f"{(horas_produccion.lt(0) | horas_produccion.gt(24)).sum()} fuera de [0, 24]."
        ),
        "Interpretación / siguiente paso": "Ceros compatibles con paradas; verificar criterio de registro para valores fuera de rango."
    },
    {
        "Señal": "Petróleo y horas",
        "Evidencia": f"{controles['petroleo_con_horas_cero'].sum()} registros del diario con petróleo positivo y horas cero.",
        "Interpretación / siguiente paso": "Revisar asignación de volumen y fecha; no corregir sin evidencia."
    },
    {
        "Señal": "Clave y continuidad temporal",
        "Evidencia": (
            f"{diario.duplicated(['NPD_WELL_BORE_CODE', 'DATEPRD']).sum()} duplicados pozo-fecha; "
            f"{huecos.gt(0).sum()} pozos con días ausentes entre su inicio y fin."
        ),
        "Interpretación / siguiente paso": "Usar calendario explícito; no confundir el siguiente registro con el día siguiente."
    },
    {
        "Señal": "Coherencia de categorías",
        "Evidencia": f"{controles['production_con_tipo_WI'].sum()} registros combinan production con WI.",
        "Interpretación / siguiente paso": "Revisar tipo de pozo y contexto antes de definir filtros."
    },
])
display(sintesis)


### Conclusión y continuidad

La revisión deja señales identificadas y controles reproducibles, pero no establece todavía un dataset apto para entrenamiento. No se han aplicado exclusiones ni imputaciones.

En `02_exploracion.ipynb`, revisar primero los pozos y períodos señalados: comparar producción, horas y disponibilidad de sensores. Contrastar allí las correlaciones globales antes de seleccionar variables. La reconciliación diaria/mensual y la confirmación de unidades siguen pendientes de incorporación o documentación dentro del flujo reproducible.

Registrar abajo las decisiones de limpieza propuestas, su evidencia y los pendientes. Una señal sin explicación permanece pendiente; no se transforma automáticamente en un error confirmado.


## Notas del análisis

- Unidades y definiciones pendientes:
- Inconsistencias que necesitan revisión:
- Variables utilizables por pozo:
- Criterios de limpieza propuestos y justificación:
